# 01 — Construção da Camada Gold (local)

**O que vamos fazer:**
1. Carregar os 5 CSVs brutos
2. Investigar a coluna `rede`
3. Detectar e remover duplicatas exatas encontradas na fonte bruta
4. Juntar indicador + meta por (município, ano)
5. Calcular a diferença em relação à meta e uma classificação
6. Validar qualidade e salvar o resultado


In [11]:
import pandas as pd

RAW_DIR = "../data/raw/"
GOLD_DIR = "../data/gold/"

# Mapeamento de código de rede de ensino -> nome legível
# (0=Total, 1 e 4=Federal, 2=Municipal, 3=Estadual, 5=Privada)
REDE_MAP = {0: "Total", 1: "Federal", 2: "Municipal", 3: "Estadual", 4: "Federal", 5: "Privada"}

## 1.1 Carregar e tipar os 5 CSVs brutos

In [12]:
indicador_municipio = pd.read_csv(RAW_DIR + "indicador_municipio.csv")
meta_municipio = pd.read_csv(RAW_DIR + "meta_alfabetizacao_municipio.csv")

# Garantir que ano e rede são números inteiros (às vezes o pandas lê como texto)
for col in ["ano", "rede"]:
    indicador_municipio[col] = pd.to_numeric(indicador_municipio[col], errors="coerce").astype("Int64")
meta_municipio["ano"] = pd.to_numeric(meta_municipio["ano"], errors="coerce").astype("Int64")

print("indicador_municipio:", indicador_municipio.shape)
print("meta_municipio:", meta_municipio.shape)
indicador_municipio.head()

indicador_municipio: (23995, 15)
meta_municipio: (10704, 13)


,ano,id_municipio,serie,rede,taxa_alfabetizacao,media_portugues,proporcao_aluno_nivel_0,proporcao_aluno_nivel_1,proporcao_aluno_nivel_2,proporcao_aluno_nivel_3,proporcao_aluno_nivel_4,proporcao_aluno_nivel_5,proporcao_aluno_nivel_6,proporcao_aluno_nivel_7,proporcao_aluno_nivel_8
0,2023,1100031,2,3,69.10,767.8763,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,2023,1100072,2,3,58.20,747.8918,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,2023,1100189,2,5,69.73,762.4062,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,2023,1101609,2,3,50.70,745.6802,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,2023,1101807,2,3,55.69,752.3724,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


## 1.2 Coluna `rede`

Por tipo de rede escolar

In [13]:
print(indicador_municipio["rede"].value_counts())
print()
print(indicador_municipio["rede"].map(REDE_MAP).value_counts())

rede
3    10896
5    10466
2     2235
0      398
Name: count, dtype: Int64

rede
Estadual     10896
Privada      10466
Municipal     2235
Total          398
Name: count, dtype: int64


## 1.3 Duplicatas entre Estadual e Privada

Comparar `taxa_alfabetizacao` de Estadual (rede=3)
com Privada (rede=5) pro mesmo município-ano.

In [14]:
pivot_check = indicador_municipio.pivot_table(
    index=["id_municipio", "ano"], columns="rede", values="taxa_alfabetizacao", aggfunc="first"
)

identicos = (pivot_check[3] == pivot_check[5]).sum()
total_comparaveis = pivot_check[[3, 5]].dropna().shape[0]

print(f"Municípios-ano com Estadual == Privada (idêntico): {identicos} de {total_comparaveis} "
      f"({identicos/total_comparaveis:.0%})")
print(f"\nLinhas ANTES de remover duplicatas: {len(indicador_municipio)}")

Municípios-ano com Estadual == Privada (idêntico): 8254 de 10348 (80%)

Linhas ANTES de remover duplicatas: 23995


Em ~80% dos municípios, "Estadual" e "Privada" trazem
valores **idênticos** — sinal de duplicação na fonte bruta. Vamos remover essas duplicatas antes de seguir, pra não contar a mesma observação duas vezes com rótulos de rede diferentes.

In [15]:
indicador_municipio = indicador_municipio.drop_duplicates(
    subset=["id_municipio", "ano", "taxa_alfabetizacao", "media_portugues"]
)
print(f"Linhas DEPOIS de remover duplicatas: {len(indicador_municipio)}")

Linhas DEPOIS de remover duplicatas: 15238


## 1.4 Juntar indicador + meta por (município, ano)

A coluna `rede` da tabela de meta tem um valor constante ("Municipal") em
todas as linhas — não distingue rede de verdade — então descartamos essa
coluna antes do join, e aplicamos a mesma meta municipal a todas as redes
daquele município (a meta é uma política por município, não por rede).

In [16]:
meta_para_juntar = meta_municipio.drop(columns=["rede"], errors="ignore").rename(columns={
    "taxa_alfabetizacao": "taxa_meta_base",
    "percentual_participacao": "percentual_participacao",
})
meta_para_juntar = meta_para_juntar.rename(
    columns={f"meta_alfabetizacao_{ano}": f"meta_{ano}" for ano in range(2024, 2031)}
)

gold_municipio = indicador_municipio.merge(meta_para_juntar, on=["id_municipio", "ano"], how="left")
gold_municipio["rede_label"] = gold_municipio["rede"].map(REDE_MAP)

print(gold_municipio.shape)
gold_municipio.head()

(15238, 26)


,ano,id_municipio,serie,rede,taxa_alfabetizacao,media_portugues,proporcao_aluno_nivel_0,proporcao_aluno_nivel_1,proporcao_aluno_nivel_2,proporcao_aluno_nivel_3,...,meta_2024,meta_2025,meta_2026,meta_2027,meta_2028,meta_2029,meta_2030,nivel_alfabetizacao,percentual_participacao,rede_label
0,2023,1100031,2,3,69.10,767.8763,NaN,NaN,NaN,NaN,...,70.85,72.53,74.15,75.71,77.21,78.64,80.0,3.0,90.48,Estadual
1,2023,1100072,2,3,58.20,747.8918,NaN,NaN,NaN,NaN,...,61.82,65.31,68.64,71.79,74.74,77.48,80.0,2.0,92.25,Estadual
2,2023,1100189,2,5,69.73,762.4062,NaN,NaN,NaN,NaN,...,71.37,72.95,74.48,75.95,77.36,78.71,80.0,3.0,90.35,Privada
3,2023,1101609,2,3,50.70,745.6802,NaN,NaN,NaN,NaN,...,55.53,60.25,64.80,69.09,73.07,76.71,80.0,2.0,95.00,Estadual
4,2023,1101807,2,3,55.69,752.3724,NaN,NaN,NaN,NaN,...,59.72,63.63,67.37,70.89,74.18,77.22,80.0,2.0,83.15,Estadual


## 1.5 Calcular diferença em relação à meta e classificação

In [17]:
def meta_do_ano(row):
    col = f"meta_{int(row['ano'])}"
    return row[col] if col in row and pd.notna(row["ano"]) else None

gold_municipio["meta_ano_vigente"] = gold_municipio.apply(meta_do_ano, axis=1)
gold_municipio["diferenca_meta"] = (
    gold_municipio["taxa_alfabetizacao"] - gold_municipio["meta_ano_vigente"]
).round(2)
gold_municipio["bateu_meta_2030"] = gold_municipio["taxa_alfabetizacao"] >= 80.0

def classificar(row):
    if pd.isna(row["diferenca_meta"]):
        return "Sem meta definida"
    if row["diferenca_meta"] >= 5:
        return "Acima da meta"
    if row["diferenca_meta"] >= 0:
        return "Na meta"
    if row["diferenca_meta"] >= -5:
        return "Abaixo da meta"
    return "Muito abaixo da meta"

gold_municipio["classificacao"] = gold_municipio.apply(classificar, axis=1)
gold_municipio[["id_municipio", "ano", "rede_label", "taxa_alfabetizacao", "meta_ano_vigente", "diferenca_meta", "classificacao"]].head()

,id_municipio,ano,rede_label,taxa_alfabetizacao,meta_ano_vigente,diferenca_meta,classificacao
0,1100031,2023,Estadual,69.10,NaN,NaN,Sem meta definida
1,1100072,2023,Estadual,58.20,NaN,NaN,Sem meta definida
2,1100189,2023,Privada,69.73,NaN,NaN,Sem meta definida
3,1101609,2023,Estadual,50.70,NaN,NaN,Sem meta definida
4,1101807,2023,Estadual,55.69,NaN,NaN,Sem meta definida


## 1.6 Validações

In [18]:
print("Nulos por coluna (%):")
display((gold_municipio.isna().mean() * 100).round(1).sort_values(ascending=False))

print("\nCobertura por ano:")
display(gold_municipio.groupby("ano")["id_municipio"].nunique())

print("\nDuplicatas remanescentes (id_municipio, ano, rede):",
      gold_municipio.duplicated(subset=["id_municipio", "ano", "rede"]).sum())

Nulos por coluna (%):


diferenca_meta             53.6
meta_ano_vigente           53.6
proporcao_aluno_nivel_3    50.4
proporcao_aluno_nivel_2    50.4
proporcao_aluno_nivel_5    50.4
proporcao_aluno_nivel_4    50.4
proporcao_aluno_nivel_6    50.4
proporcao_aluno_nivel_7    50.4
proporcao_aluno_nivel_1    50.4
proporcao_aluno_nivel_0    50.4
proporcao_aluno_nivel_8    50.4
meta_2024                   6.2
nivel_alfabetizacao         4.8
percentual_participacao     4.8
taxa_meta_base              4.8
meta_2028                   3.9
meta_2029                   3.9
meta_2026                   3.9
meta_2030                   3.9
meta_2025                   3.9
meta_2027                   3.9
media_portugues             0.0
serie                       0.0
rede                        0.0
ano                         0.0
id_municipio                0.0
taxa_alfabetizacao          0.0
rede_label                  0.0
bateu_meta_2030             0.0
classificacao               0.0
dtype: float64


Cobertura por ano:


ano
2023    5514
2024    5516
Name: id_municipio, dtype: int64


Duplicatas remanescentes (id_municipio, ano, rede): 0


**Nota:** 2023 não tem meta definida (as metas só existem a partir de
2024), então boa parte de `classificacao` para 2023 aparece como "Sem meta
definida".

In [19]:
display(pd.crosstab(gold_municipio["ano"], gold_municipio["classificacao"]))

classificacao,Abaixo da meta,Acima da meta,Muito abaixo da meta,Na meta,Sem meta definida
ano,,,,,
2023,0,0,0,0,7685
2024,962,2752,2390,972,477


## 1.7 Salvar Gold local

In [20]:
import os
os.makedirs(GOLD_DIR, exist_ok=True)
gold_municipio.to_parquet(GOLD_DIR + "gold_indicador_municipio.parquet", index=False)
print("Salvo em:", GOLD_DIR + "gold_indicador_municipio.parquet")
print("Shape final:", gold_municipio.shape)

Salvo em: ../data/gold/gold_indicador_municipio.parquet
Shape final: (15238, 30)
